# Entrenar un detector de cajas (YOLOX-Nano) en Colab

Para qué sirve: aprende a **marcar con una caja** lo que tú le enseñes (una clase o varias) en tus propias capturas, y
deja el modelo listo para el detector de GoondVR (`detector/models/<nombre>/`).

Qué necesitas antes: una carpeta de imágenes con sus cajas en formato YOLO (`.txt`) convertida con
`python prepare_detection.py --data mi_dataset --out mi_dataset_coco` y **comprimida en un .zip** (la carpeta
`mi_dataset_coco` entera). Está explicado paso a paso en `docs/GUIA_DETECCION.md`.

Pasos: 1) ajustes, 2) preparar, 3) datos, 4) entrenar, 5) exportar y comprobar, 6) descargar.
Antes de empezar: menú **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.

> **Aviso honesto:** este cuaderno se escribió sin poder ejecutarlo (quien lo escribió no tenía Colab ni GPU).
> Si una celda falla, el mensaje de error dice cuál y por qué; no sigas con las siguientes.
> La licencia de YOLOX es Apache-2.0 (uso libre, también comercial).
> Tus imágenes se suben a Google (Drive o Colab) mientras dura el entrenamiento.

In [ ]:
#@title 1. Ajustes { display-mode: "form" }
NOMBRE_MODELO = "mi_detector" #@param {type:"string"}
DATOS = "drive" #@param ["drive", "subir"]
RUTA_ZIP_EN_DRIVE = "/content/drive/MyDrive/goondvr/mi_dataset_coco.zip" #@param {type:"string"}
EPOCAS = 100 #@param {type:"integer"}
LOTE = 16 #@param {type:"integer"}
TAMANO = 416 #@param [320, 416, 512, 640] {type:"raw"}

import re
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,63}", NOMBRE_MODELO), "El nombre solo puede llevar letras, números, _ . -"
assert TAMANO % 32 == 0

In [ ]:
#@title 2. Preparar YOLOX (unos 2 minutos)
import os, torch
if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU: menú «Entorno de ejecución» → «Cambiar tipo de entorno» → GPU (T4) y vuelve a empezar")
print("GPU:", torch.cuda.get_device_name(0))

# YOLOX se usa desde su carpeta (sin instalarlo): solo se instalan las librerías sueltas que necesita
!([ -d /content/YOLOX ] || git clone -q --depth 1 --branch 0.3.0 https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX)
!pip install -q loguru tabulate thop psutil pycocotools opencv-python-headless scikit-image tensorboard tqdm onnx onnxruntime
!([ -s /content/yolox_nano.pth ] || wget -q -O /content/yolox_nano.pth https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth)
os.environ["PYTHONPATH"] = "/content/YOLOX"

# pesos preentrenados (COCO): se guardan otra vez como un diccionario simple para que PyTorch reciente no los rechace
try:
    ck = torch.load("/content/yolox_nano.pth", map_location="cpu", weights_only=False)
except TypeError:  # PyTorch antiguo, sin ese parámetro
    ck = torch.load("/content/yolox_nano.pth", map_location="cpu")
torch.save({"model": ck["model"]}, "/content/yolox_nano.pth")
print("listo")

In [ ]:
#@title 3. Datos
import glob, json, shutil, zipfile
if DATOS == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = RUTA_ZIP_EN_DRIVE
else:
    from google.colab import files
    subidos = files.upload()
    zip_path = next(iter(subidos))
shutil.rmtree("/content/dataset", ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content/dataset")
hits = glob.glob("/content/dataset/**/annotations/instances_train2017.json", recursive=True)
if len(hits) != 1:
    raise RuntimeError("En el .zip debe estar la carpeta que crea prepare_detection.py (con train2017/, val2017/ y annotations/)")
DATA_DIR = os.path.dirname(os.path.dirname(hits[0]))
train_ann = json.load(open(hits[0], encoding="utf-8"))
val_ann = json.load(open(os.path.join(DATA_DIR, "annotations", "instances_val2017.json"), encoding="utf-8"))
CLASES = [c["name"] for c in sorted(train_ann["categories"], key=lambda c: c["id"])]
print(f"{len(train_ann['images'])} imágenes de entrenamiento, {len(val_ann['images'])} de validación, clases: {CLASES}")
for i, n in enumerate(CLASES):
    c = sum(a["category_id"] == i + 1 for a in train_ann["annotations"])
    print(f"  {n}: {c} cajas" + ("   <- pocas: con menos de ~50 el resultado será flojo" if c < 50 else ""))
OUT_DIR = "/content/drive/MyDrive/goondvr/YOLOX_outputs" if DATOS == "drive" else "/content/YOLOX_outputs"

In [ ]:
#@title 4. Entrenar (el tiempo depende de las imágenes y las épocas; con Drive, si se corta, se puede continuar)
lo, hi = TAMANO // 32 - 3, TAMANO // 32 + 3
EXP = f"""
import torch.nn as nn
from yolox.exp import Exp as MyExp


class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth = 0.33
        self.width = 0.25
        self.input_size = ({TAMANO}, {TAMANO})
        self.test_size = ({TAMANO}, {TAMANO})
        self.random_size = ({lo}, {hi})
        self.mosaic_scale = (0.5, 1.5)
        self.enable_mixup = False
        self.data_dir = "{DATA_DIR}"
        self.train_ann = "instances_train2017.json"
        self.val_ann = "instances_val2017.json"
        self.num_classes = {len(CLASES)}
        self.max_epoch = {EPOCAS}
        self.warmup_epochs = 2
        self.no_aug_epochs = {max(1, min(15, EPOCAS // 6))}
        self.eval_interval = 5
        self.print_interval = 20
        self.save_history_ckpt = False
        self.data_num_workers = 2
        self.exp_name = "goondvr"
        self.output_dir = "{OUT_DIR}"

    def get_model(self, sublinear=False):
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        if "model" not in self.__dict__:
            from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

            in_channels = [256, 512, 1024]
            backbone = YOLOPAFPN(self.depth, self.width, in_channels=in_channels, depthwise=True)
            head = YOLOXHead(self.num_classes, self.width, in_channels=in_channels, depthwise=True)
            self.model = YOLOX(backbone, head)
        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        return self.model
"""
open("/content/exp_goondvr.py", "w").write(EXP)
os.makedirs(OUT_DIR, exist_ok=True)
CARPETA = f"{OUT_DIR}/goondvr"
ULTIMO = f"{CARPETA}/latest_ckpt.pth"
if os.path.exists(ULTIMO):
    print("Hay un entrenamiento a medias: se continúa desde él")
    try:
        ck = torch.load(ULTIMO, map_location="cpu", weights_only=False)
    except TypeError:
        ck = torch.load(ULTIMO, map_location="cpu")
    ck["best_ap"] = float(ck.get("best_ap", 0))   # un número de numpy hace que PyTorch reciente rechace el archivo
    torch.save(ck, ULTIMO)
    PUNTO_DE_PARTIDA = f"--resume -c {ULTIMO}"
else:
    PUNTO_DE_PARTIDA = "-c /content/yolox_nano.pth"
!cd /content/YOLOX && python tools/train.py -f /content/exp_goondvr.py -d 1 -b {LOTE} --fp16 {PUNTO_DE_PARTIDA}

In [ ]:
#@title 5. Exportar a ONNX y comprobar con el mismo código que usa el detector
import sys
import numpy as np
import onnxruntime as ort
import torch
from torch import nn
from PIL import Image
sys.path.insert(0, "/content/YOLOX")
from yolox.exp import get_exp
from yolox.models.network_blocks import SiLU
from yolox.utils import replace_module

CKPT = next((p for p in (f"{CARPETA}/best_ckpt.pth", f"{CARPETA}/last_epoch_ckpt.pth", f"{CARPETA}/latest_ckpt.pth") if os.path.exists(p)), None)
if CKPT is None:
    raise RuntimeError(f"No encuentro ningún punto de control en {CARPETA}: ¿terminó el entrenamiento?")
print("usando", CKPT)
exp = get_exp("/content/exp_goondvr.py")
model = exp.get_model()
try:
    ck = torch.load(CKPT, map_location="cpu", weights_only=False)
except TypeError:
    ck = torch.load(CKPT, map_location="cpu")
model.load_state_dict(ck["model"] if "model" in ck else ck)
model.eval()
model = replace_module(model, nn.SiLU, SiLU)
model.head.decode_in_inference = False   # salida cruda: el detector la decodifica él mismo
ONNX = "/content/model.onnx"
dummy = torch.randn(1, 3, TAMANO, TAMANO)
kw = dict(input_names=["images"], output_names=["output"], opset_version=11)
try:
    torch.onnx.export(model, dummy, ONNX, dynamo=False, **kw)
except TypeError:  # PyTorch sin el parámetro dynamo
    torch.onnx.export(model, dummy, ONNX, **kw)

sess = ort.InferenceSession(ONNX, providers=["CPUExecutionProvider"])
salida = sess.run(None, {"images": np.random.rand(1, 3, TAMANO, TAMANO).astype("float32") * 255})[0]
esperado = sum((TAMANO // s) ** 2 for s in (8, 16, 32))
assert salida.shape == (1, esperado, 5 + len(CLASES)), f"forma inesperada {salida.shape}"

# lo mismo que hace el detector: misma preparación de la imagen y misma decodificación
!([ -d /content/goondvr ] || git clone -q --depth 1 --branch community-modifications https://github.com/Danyel2505/goondvr.git /content/goondvr)
sys.path.insert(0, "/content/goondvr/detector")
import common as C
real = {}
for a in val_ann["annotations"]:
    real.setdefault(a["image_id"], set()).add(a["category_id"] - 1)
iguales = total = 0
for im in val_ann["images"][:100]:
    img = Image.open(f"{DATA_DIR}/val2017/{im['file_name']}").convert("RGB")
    x, r = C.prep_yolox(img, TAMANO)
    cajas = C.yolox_boxes(sess.run(None, {"images": x})[0][0], TAMANO, r, img.width, img.height, conf=0.5)
    total += 1
    iguales += {b[0] for b in cajas} == real.get(im["id"], set())
print(f"{iguales} de {total} imágenes de validación salen con exactamente las clases que marcaste (con confianza ≥ 50 %)")

In [ ]:
#@title 6. Empaquetar y descargar
sys.path.insert(0, "/content/goondvr/detector")
import common as C
carpeta = f"/content/modelo/{NOMBRE_MODELO}"
shutil.rmtree("/content/modelo", ignore_errors=True)
os.makedirs(carpeta)
shutil.copyfile(ONNX, f"{carpeta}/model.onnx")
C.write_model_folder(carpeta, {"kind": "yolox", "size": TAMANO}, CLASES)
zip_final = shutil.make_archive(f"/content/{NOMBRE_MODELO}", "zip", "/content/modelo")
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs("/content/drive/MyDrive/goondvr", exist_ok=True)
    shutil.copyfile(zip_final, f"/content/drive/MyDrive/goondvr/{NOMBRE_MODELO}.zip")
    print("copia en Drive: goondvr/" + NOMBRE_MODELO + ".zip")
print("Descomprime el .zip dentro de detector/models/ (debe quedar detector/models/" + NOMBRE_MODELO + "/model.onnx)")
from google.colab import files
files.download(zip_final)